## tl;dr

Après normalisation de la géométrie sur un diamètre de fond d'œil commun et conservation des noms cliniques valides malgré le seuil de longueur, les patients ont des scores supérieurs aux témoins pour les trois rangs (Wilcoxon apparié, p bilatérales corrigées de Holm ≤ 0,033). En revanche, les données ne montrent pas que le rang 3 est systématiquement plus tortueux que les rangs 1 et 2. L'association bilatérale est surtout visible au rang 3. La comparaison artères–veines de rang 3 est suggestive mais sous-puissante, car seules neuf paires œil/artères/veines sont disponibles. Les analyses âge et sévérité ne sont pas calculables sans `demo/clinical_metadata.csv`.


## Context & Methods

L'unité primaire est le patient. Les comparaisons patient–témoin sont appariées sur `patient_id`, œil et rang. Les deux yeux d'un patient ne sont pas traités comme deux patients indépendants pour la comparaison des rangs. Les trajectoires sont ramenées à un diamètre de fond d'œil de 1024 px avant le rééchantillonnage, le lissage et le seuil de longueur.

### Key Assumptions

- `OD_de_<id>` ou `OG_de_<id>` désigne le témoin apparié au patient `<id>`.
- `1°A`, `2°A`, `3°A` désignent les rangs artériels; `inf` et `sup` désignent les territoires inférieur et supérieur.
- Les tests bilatéraux sont les résultats principaux; les p-values directionnelles sont secondaires et correspondent aux hypothèses cliniques préspécifiées.
- Une absence de différence OD–OG ne démontre pas une équivalence. L'association OD–OG est donc aussi évaluée par Spearman.


## Data

### 1. Recalculer et structurer les vaisseaux sauvegardés

In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr

from tortuosite_score.vessels_detection.clinical_excel import (
    build_structured_vessel_data,
    _arteries_vs_veins_sheet,
    _od_vs_og_sheet,
    _patient_vs_control_sheet,
    _rank_eye_scores,
    _same_eye_rank_sheet,
)
from tortuosite_score.vessels_detection.scoring import scoring_config

project_root = Path.cwd()
if not (project_root / "demo" / "streamlit_runs").exists():
    project_root = Path.cwd().parent
run_dirs = sorted(path for path in (project_root / "demo" / "streamlit_runs").iterdir() if path.is_dir())
structured, quality = build_structured_vessel_data(run_dirs, scoring_config("local_bump"))
rank_scores = _rank_eye_scores(structured)
print({
    "runs_exploitables": int(structured["run"].nunique()),
    "vaisseaux_sauvegardes": int(len(structured)),
    "vaisseaux_eligibles_filtre_longueur": int(structured["eligible_filtre_longueur"].sum()),
    "vaisseaux_eligibles_analyse": int(structured["eligible_analyse"].sum()),
    "patients": int(structured.loc[structured["groupe"] == "patient", "patient_id"].nunique()),
})

{'runs_exploitables': 33, 'vaisseaux_sauvegardes': 360, 'vaisseaux_eligibles_filtre_longueur': 331, 'vaisseaux_eligibles_analyse': 355, 'patients': 11}


### 2. Vérifier le naming, la complétude et la résolution

In [2]:
naming_profile = (
    structured.groupby(["groupe", "type_vaisseau", "rang"], dropna=False)
    .size().rename("n_vaisseaux").reset_index()
)
display(naming_profile)

resolution_profile = (
    structured[["run", "groupe", "diametre_fond_oeil_px", "facteur_normalisation"]]
    .drop_duplicates("run")
    .groupby("groupe")
    .agg(
        n_runs=("run", "count"),
        diametre_median_px=("diametre_fond_oeil_px", "median"),
        diametre_min_px=("diametre_fond_oeil_px", "min"),
        diametre_max_px=("diametre_fond_oeil_px", "max"),
    )
)
display(resolution_profile.round(1))
display(quality["probleme"].value_counts().rename_axis("probleme").reset_index(name="n"))

,groupe,type_vaisseau,rang,n_vaisseaux
0,patient,artere,1.0,42
1,patient,artere,2.0,57
2,patient,artere,3.0,63
3,patient,artere,NaN,67
4,patient,veine,NaN,29
5,temoin,artere,1.0,22
6,temoin,artere,2.0,23
7,temoin,artere,3.0,23
8,temoin,artere,NaN,18
9,temoin,veine,NaN,16


,n_runs,diametre_median_px,diametre_min_px,diametre_max_px
groupe,,,,
patient,22,860.2,641.9,1512.0
temoin,11,3756.6,1180.9,4765.9


,probleme,n
0,rang_arteriel_non_classe,85
1,vaisseau_court_conserve_nom_valide,24
2,donnees_cliniques_manquantes,11
3,temoin_manquant,10
4,vaisseau_exclu_longueur,5
5,run_sans_etat_manuel,1
6,aucun_vaisseau_score,1
7,resolution_confondue_avec_groupe,1


## Results

### 3. Comparer les rangs au niveau patient

In [3]:
same_eye = _same_eye_rank_sheet(rank_scores)
display(same_eye.loc[same_eye["section"] == "resume"].dropna(axis=1, how="all").round(4))

,section,comparaison,n_paires,statistique_friedman,p_value,conclusion,delta_moyen,delta_median,proportion_delta_positif,ic95_delta_moyen_bas,ic95_delta_moyen_haut,taille_effet_biserielle,p_value_wilcoxon,hypothese_directionnelle,p_value_directionnelle,p_value_wilcoxon_holm,p_value_directionnelle_holm
0,resume,R1_vs_R2_vs_R3_omnibus_patient,11.0,0.7273,0.6951,pas de preuve statistique de difference global...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,resume,R3_vs_R1,11.0,NaN,NaN,pas de preuve statistique de difference,1.4021,-4.4242,0.4545,-16.5616,21.4235,-0.0606,0.8984,greater,0.5845,0.9297,0.5845
2,resume,R3_vs_R2,11.0,NaN,NaN,pas de preuve statistique de difference,8.8066,4.7017,0.5455,-7.9280,25.9260,0.2727,0.4648,greater,0.2324,0.9297,0.4648


### 4. Comparer chaque rang entre patient et témoin apparié

In [4]:
patient_control = _patient_vs_control_sheet(rank_scores)
display(patient_control.loc[patient_control["section"] == "resume"].dropna(axis=1, how="all").round(4))

,section,comparaison,n_paires,delta_moyen,delta_median,proportion_delta_positif,ic95_delta_moyen_bas,ic95_delta_moyen_haut,taille_effet_biserielle,p_value_wilcoxon,hypothese_directionnelle,p_value_directionnelle,conclusion,p_value_wilcoxon_holm,p_value_directionnelle_holm
0,resume,rang_1_patient_vs_temoin,11.0,47.1881,47.9453,0.7273,17.0241,78.5067,0.7273,0.0322,greater,0.0161,compatible avec un score plus eleve dans le pr...,0.0322,0.0161
1,resume,rang_2_patient_vs_temoin,11.0,47.7767,49.8718,0.9091,29.0731,65.1967,0.9394,0.0029,greater,0.0015,compatible avec un score plus eleve dans le pr...,0.0088,0.0044
2,resume,rang_3_patient_vs_temoin,11.0,59.0698,66.4348,0.8182,31.1322,85.4437,0.8788,0.0068,greater,0.0034,compatible avec un score plus eleve dans le pr...,0.0137,0.0068


### 5. Évaluer la différence et l'association entre OD et OG

In [5]:
od_og = _od_vs_og_sheet(rank_scores)
display(od_og.loc[od_og["section"] == "resume"].dropna(axis=1, how="all").round(4))

,section,comparaison,n_paires,delta_moyen,delta_median,proportion_delta_positif,ic95_delta_moyen_bas,ic95_delta_moyen_haut,taille_effet_biserielle,p_value_wilcoxon,hypothese_directionnelle,conclusion,p_value_wilcoxon_holm,rho_spearman,p_value_spearman,p_value_spearman_holm
0,resume,rang_1_difference_OD_vs_OG,10.0,23.7959,21.8569,0.8,-1.9376,50.2072,0.6000,0.1055,,pas de preuve statistique de difference,0.3164,NaN,NaN,NaN
1,resume,rang_1_association_OD_OG,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,association OD-OG non demontree; absence de di...,NaN,0.3212,0.3655,0.3655
2,resume,rang_2_difference_OD_vs_OG,10.0,-7.3196,-3.8118,0.3,-24.5083,7.4667,-0.3818,0.3223,,pas de preuve statistique de difference,0.6445,NaN,NaN,NaN
3,resume,rang_2_association_OD_OG,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,association OD-OG non demontree; absence de di...,NaN,0.6727,0.0330,0.0661
4,resume,rang_3_difference_OD_vs_OG,10.0,8.4321,3.8287,0.6,-10.9062,26.8490,0.2727,0.4922,,pas de preuve statistique de difference,0.6445,NaN,NaN,NaN
5,resume,rang_3_association_OD_OG,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,association positive compatible avec une tortu...,NaN,0.7818,0.0075,0.0226


### 6. Comparer artères et veines

In [6]:
arteries_veins = _arteries_vs_veins_sheet(structured)
display(arteries_veins.loc[arteries_veins["section"] == "resume"].dropna(axis=1, how="all").round(4))

,section,comparaison,n_paires,delta_moyen,delta_median,proportion_delta_positif,ic95_delta_moyen_bas,ic95_delta_moyen_haut,taille_effet_biserielle,p_value_wilcoxon,hypothese_directionnelle,p_value_directionnelle,conclusion,p_value_wilcoxon_holm,p_value_directionnelle_holm,n_arteres,n_veines,p_value_mannwhitney
0,resume,rang_1_arteres_vs_veines,8.0,5.9026,-0.6934,0.5000,-14.5353,30.1708,0.0556,0.9453,greater,0.4727,pas de preuve statistique de difference,0.9453,0.4727,NaN,NaN,NaN
1,resume,rang_2_arteres_vs_veines,9.0,12.7962,9.0806,0.5556,-9.1223,37.5354,0.3333,0.4258,greater,0.2129,pas de preuve statistique de difference,0.8516,0.4258,NaN,NaN,NaN
2,resume,rang_3_arteres_vs_veines,9.0,30.9754,18.0101,0.6667,0.6100,62.2051,0.6000,0.1289,greater,0.0645,pas de preuve statistique de difference,0.3867,0.1934,NaN,NaN,NaN
3,resume,rang_3_arteres_vs_veines_nonapparie,NaN,29.6049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pas de preuve statistique de difference,NaN,NaN,21.0,9.0,0.1476


### 7. Vérifier le biais résiduel lié au format d'acquisition

In [7]:
eligible_arteries = structured[
    (structured["eligible_analyse"])
    & (structured["type_vaisseau"] == "artere")
    & (structured["rang"].isin([1, 2, 3]))
].copy()
eye_scores = (
    eligible_arteries.groupby(["run", "groupe"])
    .apply(lambda group: (group["score"] * group["longueur"]).sum() / group["longueur"].sum(), include_groups=False)
    .rename("score_moyen_pondere").reset_index()
    .merge(
        structured[["run", "diametre_fond_oeil_px"]].drop_duplicates("run"),
        on="run",
        how="left",
    )
)
for group_name, group in eye_scores.groupby("groupe"):
    rho, p_value = spearmanr(group["diametre_fond_oeil_px"], group["score_moyen_pondere"])
    print(group_name, {"n": len(group), "rho_resolution_score": round(float(rho), 3), "p_value": round(float(p_value), 4)})

patient {'n': 22, 'rho_resolution_score': -0.449, 'p_value': 0.0361}
temoin {'n': 11, 'rho_resolution_score': -0.018, 'p_value': 0.9577}


## Takeaways

1. Le premier export mélangeait une unité pixel non comparable entre acquisitions. Cette erreur est corrigée par une normalisation géométrique avant scoring.
2. La différence patient–témoin est robuste dans cet échantillon pour les trois rangs, mais elle reste potentiellement confondue par le protocole d'acquisition: les groupes proviennent de résolutions très différentes et la normalisation ne remplace pas un plan d'acquisition harmonisé.
3. Le rang 3 n'est pas « toujours » supérieur aux rangs 1 et 2 selon le score local-bump; environ la moitié à deux tiers des patients seulement ont un delta positif selon la comparaison.
4. La question OD–OG est mieux traitée comme une question d'association que comme un simple test de différence. Le signal le plus net concerne le rang 3.
5. Le contraste artères–veines de rang 3 va dans le sens clinique, mais neuf paires seulement sont disponibles. Il faut compléter les annotations veineuses avant de conclure.
6. Il faut fournir `demo/clinical_metadata.csv` (`patient_id,age,severite`) pour les deux corrélations cliniques.
